In [ ]:
# ProjectTwoDashboard - Initial Setup - 4-14-2025, A.R.

# import Dash and JupyterDash
from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State

# Visualization
import plotly.express as px
import dash_leaflet as dl

# Utilities
import pandas as pd
import numpy as np
import base64
import os

# Importing CRUD module
from animal_shelter import AnimalShelter

# Connecting to MongoDB with full parameters (based on ModuleSix lessons learned)
username = "YOUR_USERNAME"
password = "YOUR_PASSWORD"
host = "YOUR_MONGODB_HOST"
port = 30686
shelter = AnimalShelter(username, password, host, port)

# Retrieve all animal documents
df = pd.DataFrame.from_records(shelter.read({}))

# Drop _id to prevent ObjectId issues in DataTable
if '_id' in df.columns:   # Only drop _id if it exists
    df.drop(columns=['_id'], inplace=True)

## Debug
#print(df.head())
# print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
# ** FIXED **, 4-14-2025, A.R.
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
# ** FIXED **, 4-14-2025, A.R.

app.layout = html.Div([
    
#   html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1("Project Two's Dashboard - SNHU CS 340"))), 
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height': '100px'}),
    html.P("Dashboard by A.R.", style={'fontSize': '18px', 'fontWeight': 'bold'}),
    
    html.Hr(),
    
    #FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
    # ** FIXED **, 4-12-2025, A.R. -> adding Filtering buttons
    html.Div([
        html.H4("Filter by Rescue Type"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset (Show All)', 'value': 'reset'}
                
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),
    
    html.Hr(),
    
    #FIXME: Set up the features for your interactive data table to make it user-friendly for your client
    # Data Table...** FIXED **, 4-12-2025, A.R.
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        filter_action="native",   # allowing search filters
        sort_action="native",   # allowing sorting columns
        sort_mode="multi",   # allows soring by multiple columns
        page_action="native",   # adding pagination
        page_current=0,   # staring on first page...'0'
        page_size=10,   # setting to TEN rows per page
        row_selectable="single",   # allowing one row to be selected
        selected_rows=[0],   # allows pre-selecting of first row
        selected_columns=[],   # Pre-initialize to empty list (tells Dash, yes, I want column selection to exist - but start with no columns selected)
        style_table={'overflowX': 'auto'},   # making table scrollable
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'maxWidth': '250px',
            'whiteSpace': 'normal',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'}
    ),
        

    html.Br(),
    html.Hr(),

    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
# ** FIXED **, 4-12-2025, A.R., -> adding MongoDB filtering logic...
    if filter_type == 'water':
        query = {
            "breed": {
                "$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'mountain':
        query = {
            "breed": {
                "$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}

        }
    elif filter_type == 'disaster':
        query = {
            "breed": {
                "$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}

        }
    else:
        query = {}
    
    filtered_df = pd.DataFrame.from_records(shelter.read(query))
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)
    
    return filtered_df.to_dict('records')
    

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # ** FIXED **, 4-15-2025, A.R. -> PIE CHART of Breed Distribution...CURRENTLY COMMENTED OUT.
    #the unfiltered view with the pie chart was visually...flawed (looked bad). 
    
    #OLD CODE BEGINS
    #if viewData is None or len(viewData) == 0:
        #return html.P("No data available to display the chart.")
    
    #dff = pd.DataFrame.from_dict(viewData)
    
    #fig = px.pie(
    #dff,
    #names='breed',
    #title='Breed Distribution')
    
    #return [dcc.Graph(figure=fig)]
    #OLD CODE ENDS
    
    # 4-15-2025, A.R. NEW CODE---> using bar graph instead (see above issues with pie chart)
    if viewData is None or len(viewData) == 0:
        return html.P("No data available to display the chart.")
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # count breeds and take top 15 for visual clarity
    breed_counts = dff['breed'].value_counts().nlargest(15)
    
    fig = px.bar(
        x=breed_counts.index,
        y=breed_counts.values,
        labels={'x': 'Breed', 'y': 'Count'},
        title='Top 15 Breeds (Bar Chart)',
        text_auto=True
    )
    fig.update_layout(xaxis_tickangle=-45)   # tilting names 45 degrees to combat overlapping names
    
    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]



app.run_server(debug=True)